# LeetCode 749 — Contain Virus bằng Python

- Link: https://leetcode.com/problems/contain-virus/
- Độ khó: **Hard**
- Chủ đề: Mô phỏng (simulation), Duyệt đồ thị trên lưới (DFS/BFS), Thành phần liên thông, Union-Find, Tập hợp (`set`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def containVirus(self, isInfected: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Một loại virus lan trên lưới. Mỗi ngày bạn chỉ đủ nguồn lực để **xây tường quanh một vùng nhiễm** — vùng nguy hiểm nhất (đe doạ nhiều ô sạch nhất). Các vùng còn lại lây sang ô kề. Hỏi tổng số bức tường cần xây.

Đây là bài **mô phỏng**: không có công thức thần kỳ, nhưng có rất nhiều chi tiết dễ sai — đếm "ô bị đe doạ" (không trùng) khác với đếm "số bức tường" (có trùng), vùng đã cách ly không được lây nữa, và thứ tự các bước trong một ngày. Kỹ năng rèn luyện: **tách bài lớn thành các hàm nhỏ rõ ràng** và kiểm chứng bằng một bản mô phỏng "nguyên văn".

Sau notebook này bạn sẽ:

1. Tìm các **thành phần liên thông** trên lưới bằng DFS/BFS.
2. Phân biệt **tập ô bị đe doạ** (frontier, `set`) và **số cạnh biên** (số tường).
3. Viết bản mô phỏng "nguyên văn" lưu từng bức tường làm trọng tài.
4. Viết bản tối ưu đánh dấu vùng đã cách ly bằng giá trị đặc biệt, bản dùng `set` toạ độ, và bản dùng **Union-Find**.
5. Phân tích độ phức tạp của một mô phỏng nhiều vòng.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Lưới `isInfected` kích thước `m × n`: `0` là ô sạch, `1` là ô nhiễm virus. Tường có thể xây trên **cạnh chung** giữa hai ô kề (4 hướng).

Mỗi đêm virus lây từ mỗi ô nhiễm sang mọi ô kề 4 hướng, trừ khi bị tường chặn. Mỗi ngày (trước khi lây), bạn chọn **một vùng nhiễm** (thành phần liên thông 4 hướng của các ô nhiễm còn hoạt động) **đe doạ nhiều ô sạch nhất** cho đêm tới, và xây tường bao quanh toàn bộ vùng đó. Vùng đó từ nay không lây nữa. Đề **đảm bảo không bao giờ có hoà**.

Lặp lại đến khi không còn vùng nào đe doạ ô sạch (virus bị chặn hết, hoặc phủ kín lưới). Trả về **tổng số bức tường** đã xây.

**Ví dụ 1**

```text
isInfected = [[0,1,0,0,0,0,0,1],
              [0,1,0,0,0,0,0,1],
              [0,0,0,0,0,0,0,1],
              [0,0,0,0,0,0,0,0]]
Output: 10
Ngày 1: chặn vùng bên trái (5 tường). Virus bên phải lây.
Ngày 2: chặn vùng bên phải (5 tường). Hết.
```

**Ví dụ 2**

```text
isInfected = [[1,1,1],
              [1,0,1],
              [1,1,1]]
Output: 4   (ô giữa bị đe doạ; cần 4 tường quanh nó)
```

**Ví dụ 3**

```text
isInfected = [[1,1,1,0,0,0,0,0,0],
              [1,0,1,0,1,1,1,1,1],
              [1,1,1,0,0,0,0,0,0]]
Output: 13  (bức tường bên trái chỉ còn 2 cạnh ra ngoài sau ngày 1)
```

**Ràng buộc**

- `m == isInfected.length`, `n == isInfected[i].length`, `1 <= m, n <= 50`
- `isInfected[i][j]` là `0` hoặc `1`.
- Luôn có đúng một vùng đe doạ nhiều ô sạch nhất ở mỗi ngày.

In [1]:
from typing import List
EXAMPLES = [
    ([[0, 1, 0, 0, 0, 0, 0, 1], [0, 1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0]], 10),
    ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], 4),
    ([[1, 1, 1, 0, 0, 0, 0, 0, 0], [1, 0, 1, 0, 1, 1, 1, 1, 1], [1, 1, 1, 0, 0, 0, 0, 0, 0]], 13),
]
def show(g):
    sym = {0: ".", 1: "V", 2: "#"}
    for row in g:
        print(" ".join(sym.get(v, str(v)) for v in row))
for g, e in EXAMPLES:
    show(g); print("->", e, "\n")

. V . . . . . V
. V . . . . . V
. . . . . . . V
. . . . . . . .
-> 10 

V V V
V . V
V V V
-> 4 

V V V . . . . . .
V . V . V V V V V
V V V . . . . . .
-> 13 



## 3. Kiến thức nền

### 3.1. Duyệt 4 hướng trên lưới

Mảng `DIRS = [(1,0), (-1,0), (0,1), (0,-1)]` và kiểm tra biên `0 <= r < m and 0 <= c < n`.

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(r, c, m, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            yield nr, nc

print(list(neighbors(0, 0, 3, 3)), list(neighbors(1, 1, 3, 3)))

[(1, 0), (0, 1)] [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2. Tìm thành phần liên thông bằng DFS dùng ngăn xếp

Bắt đầu từ một ô chưa thăm, đẩy vào ngăn xếp; lấy ra, thăm hàng xóm cùng loại, đẩy vào. Dùng ngăn xếp tự quản lý để tránh giới hạn đệ quy (lưới 50×50 = 2500 ô).

In [3]:
def components(grid, val=1):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    comps = []
    for r in range(m):
        for c in range(n):
            if grid[r][c] == val and not seen[r][c]:
                seen[r][c] = True
                stack, comp = [(r, c)], []
                while stack:
                    x, y = stack.pop()
                    comp.append((x, y))
                    for nx, ny in neighbors(x, y, m, n):
                        if grid[nx][ny] == val and not seen[nx][ny]:
                            seen[nx][ny] = True
                            stack.append((nx, ny))
                comps.append(comp)
    return comps

print(components(EXAMPLES[0][0]))

[[(0, 1), (1, 1)], [(0, 7), (1, 7), (2, 7)]]


### 3.3. `set` để đếm phần tử **không trùng**

Ô sạch `(r,c)` có thể kề với **nhiều** ô nhiễm của cùng một vùng. Số ô bị đe doạ = kích thước `set`; số tường = số **cặp** (ô nhiễm, ô sạch kề) — có trùng ô sạch.

In [4]:
region = [(0, 0), (0, 2), (1, 1)]      # ba ô nhiễm cùng kề ô (0,1)
threat, walls = set(), 0
for r, c in [(0, 0), (0, 2)]:
    threat.add((0, 1)); walls += 1
print("ô bị đe doạ:", len(threat), " số tường:", walls)

ô bị đe doạ: 1  số tường: 2


### 3.4. Union-Find (Disjoint Set Union) cơ bản

Hai thao tác: `find(x)` tìm đại diện của tập chứa `x` (có nén đường), `union(a, b)` gộp hai tập.

In [5]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[max(ra, rb)] = min(ra, rb)    # giữ đại diện là chỉ số nhỏ nhất

d = DSU(5); d.union(3, 4); d.union(1, 3)
print([d.find(i) for i in range(5)])

[0, 1, 2, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Một ngày gồm 4 bước

1. **Tìm các vùng** đang hoạt động (thành phần liên thông của ô `1`).
2. Với mỗi vùng tính: tập ô sạch bị đe doạ (`frontier`) và số tường cần xây (`walls`).
3. **Chặn** vùng có `|frontier|` lớn nhất: cộng `walls` vào đáp án, đánh dấu vùng đó là "đã cách ly" (không bao giờ lây nữa).
4. **Lây**: mọi vùng còn lại nhiễm sang toàn bộ `frontier` của chúng.

Dừng khi không còn vùng nào có `frontier` khác rỗng.

### 4.2. Tính tay ví dụ 3

```text
V V V . . . . . .
V . V . V V V V V
V V V . . . . . .
```

- Vùng A (vòng trái, 8 ô): frontier = ô giữa (1,1) + 3 ô cột 3 → 4 ô. Tường: 4 (quanh ô giữa) + 3 (bên phải) = 7.
- Vùng B (thanh ngang, 5 ô): frontier = 5 ô trên + 5 ô dưới + ô (1,3) = 11 ô. Tường: 5 + 5 + 1 = 11.

Ngày 1 chặn B (11 tường). A lây: (1,1), (0,3), (1,3), (2,3) thành nhiễm. Ngày 2: A mở rộng chỉ còn đe doạ (0,4) và (2,4) (vì (1,4) là ô đã cách ly) → 2 tường. Tổng `11 + 2 = 13`. ✔

In [6]:
g = [row[:] for row in EXAMPLES[2][0]]
for comp in components(g):
    front, walls = set(), 0
    for r, c in comp:
        for nr, nc in neighbors(r, c, len(g), len(g[0])):
            if g[nr][nc] == 0:
                front.add((nr, nc)); walls += 1
    print(f"vùng bắt đầu tại {comp[0]}: {len(comp)} ô, đe doạ {len(front)} ô, cần {walls} tường")

vùng bắt đầu tại (0, 0): 8 ô, đe doạ 4 ô, cần 7 tường
vùng bắt đầu tại (1, 4): 5 ô, đe doạ 11 ô, cần 11 tường


## 5. Cách 1 — Mô phỏng "nguyên văn" với tập tường tường minh

**Ý tưởng.** Làm đúng như đề mô tả, kể cả **lưu từng bức tường** (một cạnh = `frozenset` của hai ô) và khi lây thì **kiểm tra có tường chặn không**. Vùng đã cách ly đánh dấu `2` (vẫn là ô nhiễm nhưng không lây). Đây là bản chậm nhưng sát đề nhất → dùng làm **trọng tài**.

Quy ước khi có hoà (đề nói không xảy ra, nhưng test ngẫu nhiên có thể gặp): chọn vùng xuất hiện **trước** theo thứ tự quét hàng–cột. Mọi cách trong notebook dùng cùng quy ước này.

In [7]:
def contain_virus_literal(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    walls = set()
    total = 0

    def blocked(a, b):
        return frozenset((a, b)) in walls

    while True:
        regions = components(g, 1)
        info = []
        for comp in regions:
            front, edges = set(), []
            for cell in comp:
                for nb in neighbors(*cell, m, n):
                    if g[nb[0]][nb[1]] == 0 and not blocked(cell, nb):
                        front.add(nb); edges.append((cell, nb))
            info.append((comp, front, edges))
        if not info or max(len(f) for _, f, _ in info) == 0:
            break
        best = max(range(len(info)), key=lambda k: (len(info[k][1]), -k))
        comp, front, edges = info[best]
        for a, b in edges:
            walls.add(frozenset((a, b)))
        total += len(edges)
        for r, c in comp:
            g[r][c] = 2                         # đã cách ly
        newly = set()
        for k, (comp2, _, _) in enumerate(info):
            if k == best:
                continue
            for cell in comp2:
                for nb in neighbors(*cell, m, n):
                    if g[nb[0]][nb[1]] == 0 and not blocked(cell, nb):
                        newly.add(nb)
        for r, c in newly:
            g[r][c] = 1
    return total

for g, e in EXAMPLES:
    print(contain_virus_literal(g), contain_virus_literal(g) == e)

10 True
4 True
13 True


**Độ phức tạp.** Mỗi ngày tốn `O(m·n)` để tìm vùng và frontier (tra tường `O(1)` trung bình). Số ngày ≤ số vùng ban đầu cộng thêm… tối đa `O(m + n)` hoặc `O(m·n)` trong trường hợp xấu (mỗi ngày cách ly một vùng). Tổng `O((m·n)^2)` xấu nhất — vẫn chạy được với `50×50`, nhưng tốn bộ nhớ cho tập tường.

**Nút thắt / quan sát.** Ta **không cần** lưu tường! Vì:

- Tường chỉ nằm quanh vùng **đã cách ly** (giá trị `2`), và vùng đó không bao giờ lây.
- Một ô sạch bên kia tường có thể bị vùng **khác** lây vào — điều đó hợp lệ (tường chỉ chặn virus của vùng bị cách ly).

Vậy chỉ cần đánh dấu vùng cách ly là `2` (khác `0` và `1`) là đủ.

## 6. Bước đột phá

**Insight 1 — tường ⇔ đánh dấu.** Mọi bức tường đều nằm trên biên của một vùng đã cách ly. Vùng đó bị "đóng băng" (giá trị `2`), không lây, và không ai lây vào nó (nó không phải ô sạch). Do đó chặn bằng tường **tương đương** đổi các ô của vùng thành `2`. Không cần lưu tường.

**Insight 2 — đếm hai đại lượng khác nhau.**

- Chọn vùng theo **số ô sạch phân biệt** bị đe doạ → `set`.
- Đáp án cộng **số tường** = số cạnh (ô nhiễm, ô sạch) → đếm có lặp.

**Insight 3 — vùng có thể nhập nhau sau khi lây.** Vì vậy mỗi ngày phải **tìm lại** các vùng từ đầu (không giữ danh sách vùng cũ).

**Bất biến.** Đầu mỗi ngày: ô `1` = đang hoạt động, ô `2` = đã cách ly, ô `0` = sạch; tổng tường đã cộng = tổng số cạnh biên của các vùng đã cách ly tại thời điểm cách ly.

**Quy tắc nếu… thì…**

- Nếu không còn vùng nào có frontier → dừng.
- Nếu vùng có frontier lớn nhất → cộng `walls`, đổi sang `2`.
- Nếu là vùng khác → nhiễm toàn bộ frontier của nó (đổi `0` → `1`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mô phỏng với DFS, đánh dấu vùng cách ly bằng `2`

**Giải thích.** Mỗi ngày quét lưới; với mỗi ô `1` chưa thăm, DFS lấy vùng, đồng thời gom `frontier` (set) và đếm `walls`. Chọn vùng max, đổi thành `2`; các vùng còn lại nhiễm frontier.

In [8]:
def contain_virus_dfs(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    total = 0
    while True:
        seen = [[False] * n for _ in range(m)]
        regions, fronts, wall_cnt = [], [], []
        for r in range(m):
            for c in range(n):
                if g[r][c] == 1 and not seen[r][c]:
                    seen[r][c] = True
                    stack, cells, front, w = [(r, c)], [], set(), 0
                    while stack:
                        x, y = stack.pop()
                        cells.append((x, y))
                        for dx, dy in DIRS:
                            nx, ny = x + dx, y + dy
                            if 0 <= nx < m and 0 <= ny < n:
                                if g[nx][ny] == 1 and not seen[nx][ny]:
                                    seen[nx][ny] = True
                                    stack.append((nx, ny))
                                elif g[nx][ny] == 0:
                                    front.add((nx, ny))
                                    w += 1
                    regions.append(cells); fronts.append(front); wall_cnt.append(w)
        if not regions:
            break
        best = max(range(len(regions)), key=lambda k: (len(fronts[k]), -k))
        if not fronts[best]:
            break
        total += wall_cnt[best]
        for k, cells in enumerate(regions):
            if k == best:
                for x, y in cells:
                    g[x][y] = 2
            else:
                for x, y in fronts[k]:
                    g[x][y] = 1
    return total

for g, e in EXAMPLES:
    print(contain_virus_dfs(g), contain_virus_dfs(g) == e)

10 True
4 True
13 True


**Code nộp LeetCode (Cách 2).**

```python
class Solution:
    def containVirus(self, isInfected: List[List[int]]) -> int:
        g = isInfected
        m, n = len(g), len(g[0])
        DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))
        total = 0
        while True:
            seen = [[False] * n for _ in range(m)]
            regions, fronts, walls = [], [], []
            for r in range(m):
                for c in range(n):
                    if g[r][c] == 1 and not seen[r][c]:
                        seen[r][c] = True
                        stack, cells, front, w = [(r, c)], [], set(), 0
                        while stack:
                            x, y = stack.pop()
                            cells.append((x, y))
                            for dx, dy in DIRS:
                                nx, ny = x + dx, y + dy
                                if 0 <= nx < m and 0 <= ny < n:
                                    if g[nx][ny] == 1 and not seen[nx][ny]:
                                        seen[nx][ny] = True
                                        stack.append((nx, ny))
                                    elif g[nx][ny] == 0:
                                        front.add((nx, ny))
                                        w += 1
                        regions.append(cells)
                        fronts.append(front)
                        walls.append(w)
            if not regions:
                break
            best = max(range(len(regions)), key=lambda k: len(fronts[k]))
            if not fronts[best]:
                break
            total += walls[best]
            for k, cells in enumerate(regions):
                if k == best:
                    for x, y in cells:
                        g[x][y] = 2
                else:
                    for x, y in fronts[k]:
                        g[x][y] = 1
        return total
```

**Độ phức tạp.** Mỗi ngày `O(m·n)`; số ngày `D` ≤ số vùng xuất hiện, tối đa `O(m·n)` nhưng thực tế nhỏ (mỗi ngày một vùng bị loại và các vùng khác lớn lên rất nhanh). Tổng `O(D·m·n)`, xấu nhất `O((m·n)^2)` lý thuyết. Space `O(m·n)`.

### 7.2. Cách 3 — Biểu diễn bằng `set` toạ độ

**Giải thích.** Thay vì lưới số, giữ ba tập: `active` (ô nhiễm còn hoạt động), `dead` (đã cách ly). Ô sạch = trong lưới, không thuộc hai tập. Tìm vùng bằng BFS trên `active`. Cách viết này ngắn gọn và tự nhiên khi lưới thưa.

Để giữ cùng quy ước hoà, ta duyệt các ô `active` theo thứ tự `sorted` (hàng rồi cột).

In [9]:
from collections import deque

def contain_virus_sets(grid):
    m, n = len(grid), len(grid[0])
    active = {(r, c) for r in range(m) for c in range(n) if grid[r][c] == 1}
    dead = set()
    total = 0
    while active:
        unseen = set(active)
        regions = []                                # (cells, front, walls)
        for start in sorted(active):
            if start not in unseen:
                continue
            unseen.discard(start)
            q, cells, front, w = deque([start]), [], set(), 0
            while q:
                x, y = q.popleft()
                cells.append((x, y))
                for nx, ny in neighbors(x, y, m, n):
                    p = (nx, ny)
                    if p in unseen:
                        unseen.discard(p); q.append(p)
                    elif p not in active and p not in dead:
                        front.add(p); w += 1
            regions.append((cells, front, w))
        best = max(range(len(regions)), key=lambda k: (len(regions[k][1]), -k))
        if not regions[best][1]:
            break
        total += regions[best][2]
        cells_best = regions[best][0]
        active.difference_update(cells_best)
        dead.update(cells_best)
        for k, (_, front, _) in enumerate(regions):
            if k != best:
                active |= front
    return total

for g, e in EXAMPLES:
    print(contain_virus_sets(g), contain_virus_sets(g) == e)

10 True
4 True
13 True


**Code nộp LeetCode (Cách 3).**

```python
from collections import deque

class Solution:
    def containVirus(self, isInfected: List[List[int]]) -> int:
        m, n = len(isInfected), len(isInfected[0])
        active = {(r, c) for r in range(m) for c in range(n) if isInfected[r][c] == 1}
        dead = set()
        total = 0
        while active:
            unseen = set(active)
            regions = []
            for start in sorted(active):
                if start not in unseen:
                    continue
                unseen.discard(start)
                q, cells, front, w = deque([start]), [], set(), 0
                while q:
                    x, y = q.popleft()
                    cells.append((x, y))
                    for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                        if not (0 <= nx < m and 0 <= ny < n):
                            continue
                        p = (nx, ny)
                        if p in unseen:
                            unseen.discard(p)
                            q.append(p)
                        elif p not in active and p not in dead:
                            front.add(p)
                            w += 1
                regions.append((cells, front, w))
            best = max(range(len(regions)), key=lambda k: len(regions[k][1]))
            if not regions[best][1]:
                break
            total += regions[best][2]
            active.difference_update(regions[best][0])
            dead.update(regions[best][0])
            for k, (_, front, _) in enumerate(regions):
                if k != best:
                    active |= front
        return total
```

**Độ phức tạp.** Như Cách 2 về mặt tiệm cận (mỗi ngày `O(m·n log(m·n))` do `sorted`, có thể bỏ `sorted` khi đề đảm bảo không hoà). Hằng số lớn hơn mảng do băm tuple.

### 7.3. Cách 4 — Gán nhãn vùng bằng Union-Find

**Giải thích.** Mỗi ngày: tạo DSU trên `m·n` ô, `union` mọi cặp ô `1` kề nhau. Đại diện của mỗi ô `1` là **chỉ số nhỏ nhất** trong vùng (do `union` giữ min) → các vùng tự động có thứ tự giống quét hàng–cột. Sau đó một lần quét: với mỗi ô `1` và mỗi hàng xóm `0`, thêm vào `front[root]` và tăng `walls[root]`.

Cách này tách bạch "gán nhãn vùng" và "thống kê theo vùng" — kiểu tư duy hữu ích khi phải gộp vùng nhiều lần.

In [10]:
def contain_virus_dsu(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    total = 0
    while True:
        d = DSU(m * n)
        for r in range(m):
            for c in range(n):
                if g[r][c] == 1:
                    if r + 1 < m and g[r + 1][c] == 1:
                        d.union(r * n + c, (r + 1) * n + c)
                    if c + 1 < n and g[r][c + 1] == 1:
                        d.union(r * n + c, r * n + c + 1)
        front, walls, members = {}, {}, {}
        for r in range(m):
            for c in range(n):
                if g[r][c] != 1:
                    continue
                root = d.find(r * n + c)
                members.setdefault(root, []).append((r, c))
                front.setdefault(root, set()); walls.setdefault(root, 0)
                for nr, nc in neighbors(r, c, m, n):
                    if g[nr][nc] == 0:
                        front[root].add((nr, nc)); walls[root] += 1
        if not members:
            break
        roots = sorted(members)                 # thứ tự theo ô nhỏ nhất của vùng
        best = max(roots, key=lambda rt: (len(front[rt]), -rt))
        if not front[best]:
            break
        total += walls[best]
        for rt in roots:
            if rt == best:
                for r, c in members[rt]:
                    g[r][c] = 2
            else:
                for r, c in front[rt]:
                    g[r][c] = 1
    return total

for g, e in EXAMPLES:
    print(contain_virus_dsu(g), contain_virus_dsu(g) == e)

10 True
4 True
13 True


**Code nộp LeetCode (Cách 4).**

```python
class Solution:
    def containVirus(self, isInfected: List[List[int]]) -> int:
        g = isInfected
        m, n = len(g), len(g[0])
        total = 0
        while True:
            parent = list(range(m * n))
            def find(x):
                while parent[x] != x:
                    parent[x] = parent[parent[x]]
                    x = parent[x]
                return x
            def union(a, b):
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[max(ra, rb)] = min(ra, rb)
            for r in range(m):
                for c in range(n):
                    if g[r][c] == 1:
                        if r + 1 < m and g[r + 1][c] == 1:
                            union(r * n + c, (r + 1) * n + c)
                        if c + 1 < n and g[r][c + 1] == 1:
                            union(r * n + c, r * n + c + 1)
            front, walls, members = {}, {}, {}
            for r in range(m):
                for c in range(n):
                    if g[r][c] != 1:
                        continue
                    rt = find(r * n + c)
                    members.setdefault(rt, []).append((r, c))
                    fs = front.setdefault(rt, set())
                    walls.setdefault(rt, 0)
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if 0 <= nr < m and 0 <= nc < n and g[nr][nc] == 0:
                            fs.add((nr, nc))
                            walls[rt] += 1
            if not members:
                break
            best = max(members, key=lambda rt: len(front[rt]))
            if not front[best]:
                break
            total += walls[best]
            for rt in members:
                if rt == best:
                    for r, c in members[rt]:
                        g[r][c] = 2
                else:
                    for r, c in front[rt]:
                        g[r][c] = 1
        return total
```

**Độ phức tạp.** Mỗi ngày `O(m·n·α(m·n))` ≈ `O(m·n)`. Tổng như Cách 2.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái theo ngày cho ví dụ 3

| Ngày | Vùng (ô đầu) | |frontier| | tường | Chọn | Tổng |
|---|---|---|---|---|---|
| 1 | A (0,0): 8 ô | 4 | 7 | | |
| 1 | B (1,4): 5 ô | 11 | 11 | **B** | 11 |
| 2 | A (0,0): 12 ô | 2 | 2 | **A** | 13 |
| 3 | không còn vùng hoạt động | — | — | dừng | 13 |

In [11]:
def trace_contain(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    total, day = 0, 0
    while True:
        day += 1
        regs = []
        for comp in components(g, 1):
            front, w = set(), 0
            for r, c in comp:
                for nr, nc in neighbors(r, c, m, n):
                    if g[nr][nc] == 0:
                        front.add((nr, nc)); w += 1
            regs.append((comp, front, w))
        if not regs or max(len(f) for _, f, _ in regs) == 0:
            print(f"Ngày {day}: không còn vùng đe doạ -> dừng. Tổng = {total}")
            return total
        best = max(range(len(regs)), key=lambda k: (len(regs[k][1]), -k))
        print(f"Ngày {day}:")
        for k, (comp, front, w) in enumerate(regs):
            mark = "  <- CHẶN" if k == best else ""
            print(f"  vùng tại {comp[0]}: {len(comp)} ô, đe doạ {len(front)}, tường {w}{mark}")
        total += regs[best][2]
        for k, (comp, front, w) in enumerate(regs):
            if k == best:
                for r, c in comp: g[r][c] = 2
            else:
                for r, c in front: g[r][c] = 1
        show(g)

trace_contain(EXAMPLES[2][0])

Ngày 1:
  vùng tại (0, 0): 8 ô, đe doạ 4, tường 7
  vùng tại (1, 4): 5 ô, đe doạ 11, tường 11  <- CHẶN
V V V V . . . . .
V V V V # # # # #
V V V V . . . . .
Ngày 2:
  vùng tại (0, 0): 12 ô, đe doạ 2, tường 2  <- CHẶN
# # # # . . . . .
# # # # # # # # #
# # # # . . . . .
Ngày 3: không còn vùng đe doạ -> dừng. Tổng = 13


13

### 8.2. Trace ví dụ 1

In [12]:
trace_contain(EXAMPLES[0][0])

Ngày 1:
  vùng tại (0, 1): 2 ô, đe doạ 5, tường 5  <- CHẶN
  vùng tại (0, 7): 3 ô, đe doạ 4, tường 4
. # . . . . V V
. # . . . . V V
. . . . . . V V
. . . . . . . V
Ngày 2:
  vùng tại (0, 6): 7 ô, đe doạ 4, tường 5  <- CHẶN
. # . . . . # #
. # . . . . # #
. . . . . . # #
. . . . . . . #
Ngày 3: không còn vùng đe doạ -> dừng. Tổng = 10


10

## 9. Code nộp LeetCode

Bản chính: **Cách 2 (DFS + đánh dấu `2`)**.

In [13]:
class Solution:
    def containVirus(self, isInfected: List[List[int]]) -> int:
        g = [row[:] for row in isInfected]      # sao chép để không phá input khi test
        m, n = len(g), len(g[0])
        DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))
        total = 0
        while True:
            seen = [[False] * n for _ in range(m)]
            regions, fronts, walls = [], [], []
            for r in range(m):
                for c in range(n):
                    if g[r][c] == 1 and not seen[r][c]:
                        seen[r][c] = True
                        stack, cells, front, w = [(r, c)], [], set(), 0
                        while stack:
                            x, y = stack.pop()
                            cells.append((x, y))
                            for dx, dy in DIRS:
                                nx, ny = x + dx, y + dy
                                if 0 <= nx < m and 0 <= ny < n:
                                    if g[nx][ny] == 1 and not seen[nx][ny]:
                                        seen[nx][ny] = True
                                        stack.append((nx, ny))
                                    elif g[nx][ny] == 0:
                                        front.add((nx, ny))
                                        w += 1
                        regions.append(cells)
                        fronts.append(front)
                        walls.append(w)
            if not regions:
                break
            best = max(range(len(regions)), key=lambda k: (len(fronts[k]), -k))
            if not fronts[best]:
                break
            total += walls[best]
            for k, cells in enumerate(regions):
                if k == best:
                    for x, y in cells:
                        g[x][y] = 2
                else:
                    for x, y in fronts[k]:
                        g[x][y] = 1
        return total

for g, e in EXAMPLES:
    assert Solution().containVirus(g) == e
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao thuật toán đúng

Đây là mô phỏng, nên "đúng" nghĩa là **mỗi bước khớp đề**:

1. **Vùng** = thành phần liên thông 4 hướng của ô nhiễm còn hoạt động → DFS từ mỗi ô `1` chưa thăm cho ra đúng các thành phần (DFS thăm mọi ô đạt được qua các ô `1` kề).
2. **Số ô bị đe doạ** của vùng = số ô sạch **phân biệt** kề vùng → `len(front)` với `front` là `set`.
3. **Số tường** để cách ly vùng = số cạnh chung giữa ô của vùng và ô sạch. Cạnh giữa ô vùng và ô `2` không cần tường (ô `2` không lây; ô vùng sắp thành `2` cũng không lây). Cạnh giữa ô vùng và ô `1` khác **không tồn tại** (nếu có thì hai ô cùng vùng). → `w` đếm mỗi cặp (ô vùng, ô sạch kề) một lần — đúng.
4. **Tương đương tường ⇔ đánh dấu `2`** (Insight 1): một bức tường chỉ ngăn virus **đi qua cạnh đó**. Virus nằm hai bên cạnh là vùng cách ly (không bao giờ lây) và ô sạch. Ô sạch nếu sau này bị vùng khác nhiễm thì cũng không lây ngược vào ô `2` (ô `2` không sạch). Vậy bức tường không bao giờ "có tác dụng" thêm ngoài việc vùng kia bị đóng băng. Cách 1 (giữ tường thật) và Cách 2 cho cùng kết quả — test ngẫu nhiên ở mục 12 xác nhận.
5. **Lây**: mọi vùng khác nhiễm toàn bộ `front` của nó (tính **trước** khi lây — các vùng lây đồng thời, ô mới nhiễm đêm nay không lây tiếp trong cùng đêm).
6. **Dừng** khi không còn vùng hoặc mọi vùng có `front` rỗng (không thể lây nữa, đề nói khi đó không cần xây thêm).

## 11. Phân tích Time/Space

Gọi `S = m·n ≤ 2500`, `D` = số ngày.

| Cách | Mỗi ngày | Tổng | Space |
|---|---|---|---|
| 1. Nguyên văn + tập tường | `O(S)` (tra tường băm) | `O(D·S)` | `O(S + #tường)` |
| 2. DFS + đánh dấu | `O(S)` | `O(D·S)` | `O(S)` |
| 3. Set toạ độ | `O(S log S)` (sorted) | `O(D·S log S)` | `O(S)` |
| 4. Union-Find | `O(S·α)` | `O(D·S)` | `O(S)` |

Mỗi ngày ít nhất một vùng bị loại vĩnh viễn và các vùng khác hoặc lớn lên hoặc hết đe doạ → `D` ≤ số vùng từng tồn tại; thực tế `D` nhỏ (vài chục). Thử với lưới `50×50` ngẫu nhiên:

In [14]:
import random, time
rng = random.Random(749)
big = [[1 if rng.random() < 0.03 else 0 for _ in range(50)] for _ in range(50)]
for fn in [contain_virus_literal, contain_virus_dfs, contain_virus_sets, contain_virus_dsu, Solution().containVirus]:
    t = time.perf_counter(); r = fn(big); t = time.perf_counter() - t
    print(f"{getattr(fn, '__name__', ''):<24} {t*1000:8.1f} ms -> {r}")

contain_virus_literal         9.7 ms -> 800
contain_virus_dfs             3.4 ms -> 800
contain_virus_sets            5.2 ms -> 800
contain_virus_dsu             7.3 ms -> 800
containVirus                  3.4 ms -> 800


## 12. Bộ test

In [15]:
ALL = [contain_virus_literal, contain_virus_dfs, contain_virus_sets, contain_virus_dsu, Solution().containVirus]

EDGE = [
    ([[0]], 0),
    ([[1]], 0),
    ([[1, 1], [1, 1]], 0),                   # phủ kín, không cần tường
    ([[1, 0]], 1),
    ([[0, 1, 0]], 2),
    ([[1, 0, 0, 0, 0]], 1),
    ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 4),
]
for fn in ALL:
    for g, e in EXAMPLES + EDGE:
        assert fn(g) == e, (fn, g, fn(g), e)
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


In [16]:
rng = random.Random(7490)
for _ in range(600):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    p = rng.choice([0.1, 0.2, 0.35])
    g = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    ref = contain_virus_literal(g)
    for fn in ALL[1:]:
        assert fn(g) == ref, (fn, g, fn(g), ref)
print("600 lưới ngẫu nhiên: mọi cách khớp bản mô phỏng nguyên văn")

600 lưới ngẫu nhiên: mọi cách khớp bản mô phỏng nguyên văn


In [17]:
# Input không bị thay đổi bởi các hàm (quan trọng khi test nhiều lần)
g = [row[:] for row in EXAMPLES[2][0]]
for fn in ALL:
    fn(g)
assert g == EXAMPLES[2][0]
print("Không hàm nào sửa input")

Không hàm nào sửa input


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chọn vùng theo **số tường** thay vì số ô bị đe doạ

**Code sai:** `best = argmax(walls)`.

**Vì sao sai:** một vùng hình vòng quanh một ô sạch có nhiều tường (4) nhưng chỉ đe doạ 1 ô. Đề yêu cầu theo **số ô sạch bị đe doạ**.

**Code đúng:** `best = argmax(len(front))`.

In [18]:
def choose_by_walls_wrong(grid):
    g = [row[:] for row in grid]; m, n = len(g), len(g[0]); total = 0
    while True:
        regs = []
        for comp in components(g, 1):
            front, w = set(), 0
            for r, c in comp:
                for nr, nc in neighbors(r, c, m, n):
                    if g[nr][nc] == 0: front.add((nr, nc)); w += 1
            regs.append((comp, front, w))
        if not regs or max(len(f) for _, f, _ in regs) == 0:
            return total
        best = max(range(len(regs)), key=lambda k: (regs[k][2], -k))     # SAI
        total += regs[best][2]
        for k, (comp, front, _) in enumerate(regs):
            if k == best:
                for r, c in comp: g[r][c] = 2
            else:
                for r, c in front: g[r][c] = 1

rng2 = random.Random(5)
for _ in range(3000):
    m, n = rng2.randint(2, 6), rng2.randint(2, 6)
    g = [[1 if rng2.random() < 0.3 else 0 for _ in range(n)] for _ in range(m)]
    if choose_by_walls_wrong(g) != Solution().containVirus(g):
        show(g); print("sai:", choose_by_walls_wrong(g), " đúng:", Solution().containVirus(g)); break

. . . .
. V . .
. . V .
V . . V
V V . .
V . V .
sai: 16  đúng: 8


### Lỗi 2 — Đếm tường bằng `len(front)` (bỏ trùng)

**Code sai:** cộng `len(front)` vào đáp án.

**Vì sao sai:** ví dụ 2: vùng vòng quanh ô giữa có `front = {(1,1)}` (1 ô) nhưng cần **4** tường.

**Code đúng:** đếm từng cạnh.

In [19]:
g = EXAMPLES[1][0]
comp = components(g, 1)[0]
front = {nb for cell in comp for nb in neighbors(*cell, 3, 3) if g[nb[0]][nb[1]] == 0}
edges = sum(1 for cell in comp for nb in neighbors(*cell, 3, 3) if g[nb[0]][nb[1]] == 0)
print("sai (len(front)):", len(front), " đúng (số cạnh):", edges)

sai (len(front)): 1  đúng (số cạnh): 4


### Lỗi 3 — Vùng đã cách ly vẫn được coi là ô nhiễm hoạt động

**Code sai:** cách ly bằng cách... không làm gì, chỉ cộng tường → hôm sau vùng đó lại được chọn/lây.

**Code đúng:** đổi sang giá trị khác (`2`) để DFS bỏ qua.

### Lỗi 4 — Lây "dây chuyền" trong cùng một đêm

**Code sai:** vừa duyệt vừa đổi `0 → 1` ngay trong lưới, khiến ô vừa nhiễm lại được dùng để lây tiếp (hoặc làm DFS gộp vùng sai).

**Code đúng:** tính hết `front` của mọi vùng **trước**, rồi mới đổi.

In [20]:
def spread_chain_wrong(g):
    g = [row[:] for row in g]; m, n = len(g), len(g[0])
    for r in range(m):
        for c in range(n):
            if g[r][c] == 1:
                for nr, nc in neighbors(r, c, m, n):
                    if g[nr][nc] == 0:
                        g[nr][nc] = 1                     # SAI: ô mới nhiễm sẽ lây tiếp khi quét tới
    return g

def spread_right(g):
    g2 = [row[:] for row in g]; m, n = len(g), len(g[0])
    for r in range(m):
        for c in range(n):
            if g[r][c] == 1:
                for nr, nc in neighbors(r, c, m, n):
                    if g[nr][nc] == 0:
                        g2[nr][nc] = 1
    return g2

print("sai:"); show(spread_chain_wrong([[1, 0, 0, 0]]))
print("đúng (một đêm chỉ lây 1 ô):"); show(spread_right([[1, 0, 0, 0]]))

sai:
V V V V
đúng (một đêm chỉ lây 1 ô):
V V . .


## 14. Mẹo & nhận dạng pattern

- **Mô phỏng theo vòng (ngày/đêm)** → viết rõ các bước của một vòng; tính mọi thay đổi **trước**, áp dụng **sau** (tránh hiệu ứng dây chuyền).
- **Thành phần liên thông trên lưới** → DFS/BFS bằng ngăn xếp/hàng đợi, hoặc Union-Find.
- **"Đếm ô phân biệt" vs "đếm cạnh"** → `set` cho ô, biến đếm cho cạnh.
- **Trạng thái "đã xử lý"** → dùng giá trị đặc biệt trong lưới (`2`, `-1`) thay cho cấu trúc phụ.
- Bài liên quan: *Number of Islands (200)*, *Island Perimeter (463)* (đếm cạnh biên), *Rotting Oranges (994)* (lây theo đợt), *Making A Large Island (827)*.

## 15. Biến thể

1. **Chỉ đếm số ngày** đến khi virus bị chặn hết.
2. **Đếm số ô sạch còn lại** cuối cùng (ô được cứu).
3. **Mỗi ngày chặn được `k` vùng**: chọn `k` vùng có frontier lớn nhất.
4. **Lây 8 hướng**: đổi `DIRS`; số tường vẫn đếm theo cạnh (4 hướng) hay theo… cần định nghĩa lại.
5. **Island Perimeter (463)**: đếm cạnh biên của một vùng — chính là phần "đếm tường".

In [21]:
# Biến thể 1 + 2: số ngày và số ô sạch còn lại
def contain_stats(grid):
    g = [row[:] for row in grid]; m, n = len(g), len(g[0]); total = days = 0
    while True:
        regs = []
        for comp in components(g, 1):
            front, w = set(), 0
            for r, c in comp:
                for nr, nc in neighbors(r, c, m, n):
                    if g[nr][nc] == 0: front.add((nr, nc)); w += 1
            regs.append((comp, front, w))
        if not regs or max(len(f) for _, f, _ in regs) == 0:
            break
        days += 1
        best = max(range(len(regs)), key=lambda k: (len(regs[k][1]), -k))
        total += regs[best][2]
        for k, (comp, front, _) in enumerate(regs):
            if k == best:
                for r, c in comp: g[r][c] = 2
            else:
                for r, c in front: g[r][c] = 1
    clean = sum(row.count(0) for row in g)
    return total, days, clean

for g, e in EXAMPLES:
    print(contain_stats(g), "(tường, ngày, ô sạch còn lại)")

(10, 2, 23) (tường, ngày, ô sạch còn lại)
(4, 1, 1) (tường, ngày, ô sạch còn lại)
(13, 2, 10) (tường, ngày, ô sạch còn lại)


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Nguyên văn | Lưu tường, kiểm tra khi lây | `O(D·S)` | `O(S + tường)` | Trọng tài, sát đề |
| 2. DFS + đánh dấu `2` | Tường ⇔ đóng băng vùng | `O(D·S)` | `O(S)` | Bản nộp chính |
| 3. Set toạ độ | active / dead / ô sạch | `O(D·S log S)` | `O(S)` | Lưới thưa, code gọn |
| 4. Union-Find | Gán nhãn rồi thống kê | `O(D·S·α)` | `O(S)` | Luyện DSU, gộp vùng |

## 17. Bài tập

**Bài 1 (đoán output).** `containVirus([[1,0,0,0,0,0,0,0,0,1]])` = ?

**Bài 2 (tính tay).** Lập bảng theo ngày (như 8.1) cho `[[1,0,1],[0,0,0],[0,0,0]]` (giả sử hoà thì chọn vùng quét trước).

**Bài 3 (điền TODO).** Hoàn thiện `region_stats(g, comp)` trả về `(len(front), walls)` của một vùng.

**Bài 4 (tự viết lại).** Viết lại Cách 2 trong hàm `my_contain(grid)`.

**Bài 5 (mini challenge).** Viết `island_perimeter(grid)` (LeetCode 463): tổng số cạnh của các ô `1` kề với ô `0` hoặc biên lưới.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def region_stats(g, comp):
    # TODO: duyệt mọi ô của comp, mọi hàng xóm hợp lệ; ô 0 -> thêm vào front, tăng walls
    pass

# Bài 4 — TODO
def my_contain(grid):
    # TODO: lặp ngày: tìm vùng, chọn max |front|, cộng walls, đánh dấu 2, lây
    pass

# Bài 5 — TODO
def island_perimeter(grid):
    # TODO: mỗi ô 1 đóng góp 4 - (số hàng xóm là 1)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: hai vùng đơn, mỗi vùng đe doạ 1 ô.
- Mức 2: hoà (đề gốc không cho hoà — đây là bài tập), theo quy ước chọn vùng trái: 1 tường; vùng phải lây sang cột 8.
- Mức 3: ngày 2: vùng phải (2 ô) đe doạ 1 ô → 1 tường. Tổng **2**.

**Bài 2.**
- Mức 1: hai vùng `(0,0)` và `(0,2)`, mỗi vùng đe doạ 2 ô — hoà, chọn `(0,0)`: 2 tường.
- Mức 2: vùng `(0,2)` lây sang `(0,1)`, `(1,2)` → vùng mới `{(0,1),(0,2),(1,2)}` đe doạ `(1,1)` và `(2,2)` (ô `(0,0)` đã là `2` nên không tính).
- Mức 3: ngày 2: frontier = `{(1,1), (2,2)}` nhưng tường = **3** (cạnh `(0,1)–(1,1)`, `(1,2)–(1,1)`, `(1,2)–(2,2)`). Tổng `2 + 3 = 5`.

**Bài 3.**
- Mức 1: dùng `neighbors`.
- Mức 2: `front` là `set`.
- Mức 3: `walls` tăng mỗi lần gặp ô `0` (kể cả trùng).

**Bài 4.**
- Mức 1: tách hàm tìm vùng.
- Mức 2: dừng khi không có vùng hoặc frontier max rỗng.
- Mức 3: lây sau khi đã thống kê xong mọi vùng.

**Bài 5.**
- Mức 1: mỗi ô `1` có 4 cạnh.
- Mức 2: mỗi hàng xóm `1` "che" một cạnh.
- Mức 3: tổng `Σ (4 - số hàng xóm 1)`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1, 2
assert Solution().containVirus([[1, 0, 0, 0, 0, 0, 0, 0, 0, 1]]) == 2
assert Solution().containVirus([[1, 0, 1], [0, 0, 0], [0, 0, 0]]) == 5

# Bài 3
def region_stats_ans(g, comp):
    m, n = len(g), len(g[0]); front, w = set(), 0
    for r, c in comp:
        for nr, nc in neighbors(r, c, m, n):
            if g[nr][nc] == 0:
                front.add((nr, nc)); w += 1
    return len(front), w

assert region_stats_ans(EXAMPLES[1][0], components(EXAMPLES[1][0])[0]) == (1, 4)
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_contain_ans(grid):
    g = [row[:] for row in grid]; m, n = len(g), len(g[0]); total = 0
    while True:
        regs = []
        for comp in components(g, 1):
            front = {nb for cell in comp for nb in neighbors(*cell, m, n) if g[nb[0]][nb[1]] == 0}
            w = sum(1 for cell in comp for nb in neighbors(*cell, m, n) if g[nb[0]][nb[1]] == 0)
            regs.append((comp, front, w))
        if not regs:
            return total
        best = max(range(len(regs)), key=lambda k: (len(regs[k][1]), -k))
        if not regs[best][1]:
            return total
        total += regs[best][2]
        for k, (comp, front, _) in enumerate(regs):
            for r, c in (comp if k == best else front):
                g[r][c] = 2 if k == best else 1

# Bài 5
def island_perimeter_ans(grid):
    m, n = len(grid), len(grid[0]); p = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1:
                p += 4 - sum(grid[nr][nc] == 1 for nr, nc in neighbors(r, c, m, n))
    return p

assert island_perimeter_ans([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 16
assert island_perimeter_ans([[1]]) == 4
rng = random.Random(77)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [[1 if rng.random() < 0.25 else 0 for _ in range(n)] for _ in range(m)]
    assert my_contain_ans(g) == contain_virus_literal(g)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template một ngày mô phỏng:**

```python
while True:
    regions = []                      # (cells, front_set, walls)
    for mỗi ô 1 chưa thăm: DFS -> cells; với hàng xóm 0: front.add, walls += 1
    if not regions or max |front| == 0: break
    best = argmax |front|
    total += walls[best]
    đánh dấu cells[best] = 2
    các vùng khác: front -> 1         # sau khi đã tính xong mọi vùng
```

**Checklist trước khi submit:**

- [ ] Chọn vùng theo **số ô sạch phân biệt** bị đe doạ.
- [ ] Cộng **số cạnh** (tường), không phải số ô.
- [ ] Vùng cách ly đánh dấu khác (`2`) để không lây/chọn lại.
- [ ] Tính frontier mọi vùng **trước**, lây **sau**.
- [ ] Tìm lại vùng **mỗi ngày** (vùng có thể nhập nhau).
- [ ] Dừng khi không còn vùng đe doạ.
- [ ] DFS bằng ngăn xếp (tránh giới hạn đệ quy với 2500 ô).